# Putting many TinyRNNs onto one GPU

Here we investigate a potential speed-up obtained by training many TinyRNNs with the same training data. This means we can run all hyperparameters on the same loop. Could be an upwards of 100x speed up.

In [1]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
from NM_TinyRNN.code.models import training_fast as speedrun
from NM_TinyRNN.code.models import datasets as ds
from NM_TinyRNN.code.models import rnns
# write some code to further parallelise the training and test it here
from NM_TinyRNN.code.models import nested_cv as nc
from NM_TinyRNN.code.models import nested_cv_io as save_data
from NM_TinyRNN.code.models import nested_jobs

import numpy as np
import pandas as pd
import torch #for testing a few things
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from importlib import reload


CODE_DIR = Path('.') ## OBS THIS MAY NEED TO BE ADJUSTED!
SAVE_PATH = CODE_DIR/'NM_TinyRNN/data/rnns'
DATA_PATH = Path('./NM_TinyRNN/data/AB_behaviour/')

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

# let's test some code!
test_data_path = DATA_PATH / "WS16"
test_save_path  = './NM_TinyRNN/data/rnns/gpu_test'
reload(save_data)
reload(ds)
reload(nc)
reload(speedrun)
reload(rnns)
# we use a trainer with ensemble model training across hyperparameters
trainer = speedrun.TrainerGPU(weight_seeds = list(range(1,11)),
                        sparsity_lambdas = [1,0.1,1e-2,1e-3,1e-4,1e-5,1e-6],
                        energy_lambdas = [1,0.1,1e-2,1e-3],
                        hebbian_lambdas = [0.0])
model = rnns.TinyRNN(rnn_type = 'constGate', hidden_size = 2)
dataset = ds.AB_Dataset(test_data_path, sequence_length = 64)
## You may test the trainer class to fit a single model:
#final_state_dict, config = trainer.fit(model, dataset)


Sequence length 64 excludes 7.1% of trials


In [3]:
#Here we've got code to run all inner loops in parallel,  
splits = nc.nested_cv_splits(dataset)
trials_df = save_data.get_model_trial_by_trial_df(model, dataset, splits['inner_folds'][0])
outer_results = nc.run_outer_fold(model, dataset,
                                  outer_loop_number = 1,
                                  n_outer_loops = 10,
                                  save_path = test_save_path, 
                                  trainer_kwargs = {'sparsity_lambdas':[1e-1,1e-2,1e-3,1e-4,1e-5],
                                                    'energy_lambdas':[0.1,1e-2,1e-3]})
print([d['val_loss'] for d in outer_results['inner_results']])


[outer 1/10]  outer eval: 11 blocks  |  9 inner folds  |  saving to NM_TinyRNN/data/rnns/gpu_test
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu


 18%|█▊        | 177/1000 [01:31<07:06,  1.93it/s]

Search complete. Best model index: 118. Val. loss: 0.5071502923965454
  Saved outer_fold_1/inner_fold_1 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_1  (val_loss=0.5072, eval_loss=0.6200)
Parallelizing 150 models on cpu


 66%|██████▋   | 663/1000 [05:21<02:30,  2.24it/s]

Search complete. Best model index: 115. Val. loss: 0.4809551239013672
  Saved outer_fold_1/inner_fold_0 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_0  (val_loss=0.4810, eval_loss=0.6425)


...

Search complete. Best model index: 121. Val. loss: 0.5129900574684143
  Saved outer_fold_1/inner_fold_7 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_7  (val_loss=0.5130, eval_loss=0.6311)


 74%|███████▍  | 738/1000 [05:55<01:56,  2.24it/s]

Search complete. Best model index: 110. Val. loss: 0.4470858573913574
  Saved outer_fold_1/inner_fold_3 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_3  (val_loss=0.4471, eval_loss=0.6504)


 75%|███████▌  | 751/1000 [06:01<01:59,  2.08it/s]


[0.4809551239013672, 0.5071502923965454, 0.48998168110847473, 0.4470858573913574, 0.47724905610084534, 0.4774588346481323, 0.4731287658214569, 0.5129900574684143, 0.47384148836135864]


In [ ]:
test_df = nested_jobs.run_training(overwrite=True, test= True)

Submitting model training for WS20 to HPC
Submitted batch job 2766870
Submitting model training for WS20 to HPC
Submitted batch job 2766871
Submitting model training for WS20 to HPC
Submitted batch job 2766872
Submitting model training for WS20 to HPC
Submitted batch job 2766873
Submitting model training for WS14 to HPC
Submitted batch job 2766874
Submitting model training for WS14 to HPC
Submitted batch job 2766875
Submitting model training for WS14 to HPC
Submitted batch job 2766876
Submitting model training for WS14 to HPC
Submitted batch job 2766877
Submitting model training for WS08 to HPC
Submitted batch job 2766878
Submitting model training for WS08 to HPC
Submitted batch job 2766879
Submitting model training for WS08 to HPC
Submitted batch job 2766880
Submitting model training for WS08 to HPC
Submitted batch job 2766881
Submitting model training for WS01 to HPC
Submitted batch job 2766882
Submitting model training for WS01 to HPC
Submitted batch job 2766883
Submitting model tra

Search complete. Best model index: 144. Val. loss: 0.47724905610084534
  Saved outer_fold_1/inner_fold_4 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_4  (val_loss=0.4772, eval_loss=0.6510)
Search complete. Best model index: 111. Val. loss: 0.4774588346481323
  Saved outer_fold_1/inner_fold_5 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_5  (val_loss=0.4775, eval_loss=0.6499)
Search complete. Best model index: 114. Val. loss: 0.47384148836135864
  Saved outer_fold_1/inner_fold_8 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_8  (val_loss=0.4738, eval_loss=0.6512)
